# 第36课 · 窗函数（window function）原理 — 矩形窗的旁瓣（sidelobe）泄漏，Hann / Hamming / Blackman 对比

**今日目标**：先比较 Rectangular / Hann / Hamming / Blackman 的形状，再看它们如何改变信号边缘；最后阅读 `io.py` 与 `windows.py`，把概念对应到 Aurora 源码。

**目标**：比较 Rectangular / Hann / Hamming / Blackman 四种窗函数（window function）的时域形状与频率响应，理解矩形窗旁瓣（sidelobe）泄漏的来源，以及平滑窗如何压制泄漏。

**为什么对 Aurora 重要**：STFT（短时傅里叶变换）的每一帧都乘以 Hann 窗；`src/aurora/audio/windows.py` 就是这节课的直接实现，选错窗函数会导致频谱泄漏污染梅尔特征。

← **上一课**　[L35 · 欧拉公式遇见 FFT](L35_euler_fft.ipynb)

> 上节课学习了 **欧拉公式遇见 FFT**：e^{-2πikn/N} 是什么，旋转因子可视化。  
> 本课将探讨 **窗函数原理**。

## 本课剧情：为什么录音要"淡入淡出"？

混音师在剪辑一段录音时，常常会在片段两端做"淡入淡出"（fade in / fade out）——音量从 0 缓慢升到最大，再缓慢降回 0。

这不只是审美，而是**数学必要性**：如果直接截取一段信号，边缘处幅值会突然从某个值跳到 0——相当于乘了一个矩形函数。FFT 不知道"这是截取边界"，会把这个跳变当成真实的高频成分，在频谱上产生**旁瓣（sidelobe）污染**。

**窗函数（window function）**解决这个问题：一串与信号等长的权重，两端接近 0、中间接近 1。信号逐点乘上这些权重，边缘被"温柔地压低"，FFT 看到的就是平滑过渡，而非突变。

三种常用窗：

| 窗 | 公式核心 | 旁瓣抑制 | 主瓣宽度 |
|---|---|---|---|
| **矩形（Rectangular）** | w[k]=1 | 差（-13 dB） | 最窄 |
| **Hann** | 0.5·(1-cos(2πk/N)) | 好（-31 dB） | 中等 |
| **Hamming** | 0.54-0.46·cos(2πk/N) | 较好（-41 dB） | 中等 |
| **Blackman** | 0.42-0.5·cos+0.08·cos(4π) | 很好（-58 dB） | 最宽 |

本节任务：实现 `describe_window(name, w)`，读取窗数组的首尾值和峰值位置，验证三种窗的形状差异。

## 1. 先看窗函数本身

窗函数是一串和信号等长的权重。把信号逐点乘上这些权重，片段边缘会被压低，中间部分保留得更多。

- Rectangular：所有位置都是 1，相当于不处理边缘。
- Hann / Blackman：两端接近 0，边缘最柔和。
- Hamming：两端不为 0，保留一点边缘能量。

用四个数字就能描述一个窗的边界行为：首端值 `w[0]`、尾端值 `w[-1]`、峰值 `w.max()`、峰值位置 `argmax(w)`。Hann / Blackman 首尾接近 0，把边缘压至几乎为零；Hamming 首尾约 0.08，保留少量边缘能量；Rectangular 首尾为 1，完全不做压制。边界值越接近 0，频谱旁瓣越低，但主瓣（main lobe）越宽——旁瓣抑制与频谱分辨率之间不可避免的权衡。

## 实验入口：把声音拆成可观察的数组

这里用64个采样点、采样率（sample rate，sr）200 Hz的短正弦，让 `signal * window` 的结果可以直接打印，便于确认边缘值是否被压低。

In [ ]:
import numpy as np
from aurora.audio import hann, hamming, blackman
print('就绪')

## 动手观察：序列怎样一步步变成信号

改 `sample_rate` 或 `duration`，观察采样点数 `N = int(sample_rate * duration)` 如何变化；下一格的窗权重数组长度将与 `N` 保持一致。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：比较几种窗函数的边缘

打印 `w[:3]` 和 `w[-3:]`，直接看矩形窗边缘值为 1.0 而 Hann 窗边缘接近 0 这一差异。

### Periodic 与 Symmetric Hann：Aurora 和 NumPy 的差异

同一个 Hann 公式有两种常见约定：

| | `np.hanning(N)` (symmetric) | `aurora.hann(N)` (periodic / DFT-even) |
|---|---|---|
| 公式 | `w[n] = 0.5(1 - cos(2πn/(N-1)))` | `w[n] = 0.5(1 - cos(2πn/N))` |
| 首端 `w[0]` | 0 | 0 |
| 尾端 `w[-1]` | ≈ 0 | ≈ 0.095（N=10）/ 0.0024（N=64） |
| 峰值位置 | `N//2 - 1` = 4（N=10） | `N//2` = 5（N=10） |
| 适用场景 | FIR 滤波器设计 | STFT 无缝拼帧（Aurora 默认）|

本课的代码实验（cell 9、18、26）用 `np.hanning`/`np.hamming` 演示对称窗的直观形状；Aurora 源码（`aurora.audio.windows`）和练习（cell 14）使用 `periodic=True` 版本。两套数值不同，但物理意义相同——都是压低边缘能量。

In [ ]:
import numpy as np

N = 16
windows = {
    'rectangular': np.ones(N),
    'hann': np.hanning(N),
    'hamming': np.hamming(N),
}

for name, w in windows.items():
    print(name)
    print('  first 5 =', np.round(w[:5], 3))
    print('  last  5 =', np.round(w[-5:], 3))


## 2. ✏️ 实现 `describe_window(name, w)`

**手算 Hann 窗（N=8）的关键点**：

公式：`w[k] = 0.5 · (1 - cos(2π·k / (N-1)))`，k = 0, 1, ..., N-1

| k | 角度 2πk/7 | cos(...) | w[k] |
|---|---|---|---|
| 0 | 0 | 1 | **0**（左端=0） |
| 1 | 2π/7 | 0.6235 | 0.1883 |
| 3 | 6π/7 | -0.9009 | **0.9505**（接近峰值） |
| 7 | 2π | 1 | **0**（右端=0） |

**规律**：
- 对称窗：`w[0] ≈ w[-1] ≈ 0`，峰值在 `N//2 - 1` 附近
- Hamming：首尾不为 0（≈ 0.08），保留少量边缘能量
- 矩形窗：`w[0] = w[-1] = 1.0`（无淡化）

`describe_window` 的返回值应是一个字典，至少包含：
- `"first"`: `w[0]`
- `"last"`: `w[-1]`
- `"peak"`: `w.max()`
- `"peak_idx"`: `int(np.argmax(w))`

### 写代码前，先把变量表补完整

写 `describe_window` 前明确三件事：
- 输入：`w`（shape `(N,)` 的窗权重数组，值域 0~1）
- 关键步骤：取 `w[0]`、`w[-1]`、`w.max()`、`int(np.argmax(w))`
- 返回：四元组 `(first, last, peak, peak_idx)`，依次是首端值、尾端值、峰值、峰值位置索引

In [ ]:
def describe_window(name, w):
    # ✏️ TODO: 打印 name、长度 len(w)、首尾值、峰值与峰值下标
    print(name, '...')  # ← 改这里，打印有用信息
    raise NotImplementedError("TODO: 计算并返回窗函数统计字典 {name, peak_sidelobe_dB, mainlobe_width}")


## 3. 在三种窗上调用 + 自动检查

In [ ]:
N = 64
windows = {'Hann': hann(N), 'Hamming': hamming(N), 'Blackman': blackman(N)}
all_ok = True
for name, w in windows.items():
    assert w.shape[0] == N
    try:
        result = describe_window(name, w)
    except NotImplementedError:
        print('⬜ 未实现')
        all_ok = False
        break
    if result is None:
        print('⬜ 未实现')
        all_ok = False
        break
    first, last, peak, peak_idx = result
    assert abs(first - w[0]) < 1e-9, f'{name}: first 值不对'
    assert abs(last - w[-1]) < 1e-9, f'{name}: last 值不对'
    assert abs(peak - w.max()) < 1e-9, f'{name}: peak 值不对'
    assert peak_idx == int(np.argmax(w)), f'{name}: peak_idx 不对'

if all_ok:
    assert abs(hann(N)[0]) < 1e-9, 'Hann 窗首端应为 0（periodic DFT-even 窗；尾端 w[-1] ≈ 0.095 for N=64，不为 0）'
    print('\n✅ 窗函数性质核对完毕。')

## 4. 画出三种窗对比

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
for name, w in windows.items():
    plt.plot(w, label=name)
plt.title(f'Window functions (N={N})')
plt.xlabel('sample'); plt.ylabel('weight')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4.1 加窗（windowing）前后：同一段信号，不同边缘

下面保留同一段正弦，只改变窗。先看时域（time domain）边缘，再看频域（frequency domain）里主峰旁边的能量。


In [ ]:
import numpy as np, matplotlib.pyplot as plt

N = 64
n = np.arange(N)
signal = np.sin(2*np.pi*5.5*n/N)  # 非整数周期，边缘会接不上
windows = {
    'rectangular': np.ones(N),
    'hann': np.hanning(N),
    'hamming': np.hamming(N),
    'blackman': np.blackman(N),  # 增补：与标题一致
}

plt.figure(figsize=(9, 3.2))
for name, w in windows.items():
    plt.plot(signal * w, label=name)
plt.title('same signal × different windows')
plt.legend(); plt.grid(True, alpha=.25); plt.show()

plt.figure(figsize=(9, 3.2))
for name, w in windows.items():
    spectrum = np.abs(np.fft.rfft(signal * w))
    plt.plot(spectrum / spectrum.max(), label=name)
plt.title('normalized spectrum after windowing')
plt.legend(); plt.grid(True, alpha=.25); plt.show()


## 5. 源文件阅读：把概念映射回 Aurora

完成窗函数实验后，再打开这两个文件：

- `src/aurora/audio/io.py`：`sine` / `chirp` / `read_wav` / `write_wav`
- `src/aurora/audio/windows.py`：`hann` / `hamming` / `blackman`

阅读时只对照三件事：函数输入、返回数组、数组长度。


## 🎨 图示：从窗 → STFT 分帧 → 频谱 → mel

把后面 L37-L47 要做的事先看一眼，统一观感。

In [ ]:
from aurora.audio import sine
import aurora.aviz as aviz; aviz.style()
long = sine(220.0, duration=0.05, sample_rate=16000)
aviz.windows_overlay(64);

In [ ]:
aviz.framing(long, frame_len=256, hop=128, n_frames=5);   # STFT 分帧

In [ ]:
aviz.spectrogram(long);                                   # 功率谱图

In [ ]:
aviz.mel_filterbank_plot(20, 256, 16000);                 # mel 滤波器组=矩阵

In [ ]:
aviz.mel_spectrogram_plot(long, 16000, 40);               # mel 频谱图

In [ ]:
N = 16
signal = np.ones(N)
for window_name, window in [('rect', np.ones(N)), ('hann', np.hanning(N)), ('hamming', np.hamming(N)), ('blackman', np.blackman(N))]:
    shaped = signal * window
    edge_energy = shaped[0]**2 + shaped[-1]**2
    print(f'{window_name:7s} | 边缘值=({shaped[0]:.3f}, {shaped[-1]:.3f}) | 边缘能量={edge_energy:.3f}')


## 参数实验：旁瓣高度直接对比

对同一段正弦信号（频率恰好在两个 FFT 频点之间），分别用 Rectangular 和 Hann 加窗后做 FFT，观察 Rectangular 的频谱旁瓣高出主瓣约 13 dB，Hann 的旁瓣低约 31 dB。差距来自边缘处理：Rectangular 的硬截断制造虚假高频，Hann 的平滑过渡大幅压制旁瓣。这就是 STFT 几乎总是用 Hann 窗的原因。

进一步实验：把 `signal = np.sin(2*np.pi*5.5*n/N)` 中的 `5.5` 改成整数（如 `6.0`），矩形窗的旁瓣会消失——频率为整数倍时，FFT 周期边界处信号恰好连续，边缘跳变为零，旁瓣污染也随之消失。

## 本课收束

现在能用 `hann(N)`、`hamming(N)`、`blackman(N)` 生成窗权重，并用 `signal * w` 完成加窗，打印边缘值确认压低效果。这对应 Aurora STFT 流程的第一步：`aurora.audio.windows` 里的三个函数在 `stft` 分帧时被逐帧调用。下一节 L37（DFT 推导）进入 FFT 实现，今天的窗权重将作为 `xw = x * w` 直接输入离散傅里叶变换。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))


## ✏️ 白板挑战：窗函数手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：Hann 窗 `w[k] = 0.5·(1-cos(2πk/(N-1)))`，N=4 时：
- w[0] = ?
- w[1] = ?（cos(2π/3) = -0.5）
- w[3] = ?（和 w[0] 对称）

**问 2**：矩形窗与 Hann 窗的首端值（w[0]）分别是 ? 和 ?

**问 3**：Hann 窗峰值在哪个 index？（N 为偶数时，峰值在 `N//2 - 1` 还是 `N//2`？）

**问 4**：为什么窗函数能减少频谱泄漏（spectral leakage）？  
（提示：矩形截断 = 乘以矩形函数 → FFT 得到 sinc 旁瓣；Hann 渐变 → 旁瓣小得多）

**问 5**：Hann 与 Blackman 的旁瓣抑制哪个更强？代价是什么？（主瓣宽度）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：Hann(N=4)
N4 = 4
w4 = np.array([0.5*(1 - np.cos(2*np.pi*k/(N4-1))) for k in range(N4)])
assert np.isclose(w4[0], 0.0, atol=1e-12), f"w[0] 应=0，得到 {w4[0]}"
assert np.isclose(w4[1], 0.5*(1 - np.cos(2*np.pi/3)), atol=1e-12)
assert np.isclose(w4[3], 0.0, atol=1e-12), f"w[3] 应=0，得到 {w4[3]}"
print(f"Q1 ✅  Hann(N=4): {np.round(w4, 4)}")
print(f"      w[0]={w4[0]:.4f}, w[1]={w4[1]:.4f}, w[3]={w4[3]:.4f}")

# 问2：矩形首端=1，Hann首端=0
rect_first = 1.0
hann_first = 0.0
assert rect_first == 1.0 and np.isclose(hann_first, 0.0)
print(f"Q2 ✅  矩形窗 w[0]={rect_first}，Hann 窗 w[0]={hann_first}（Hann 两端压到0）")

# 问3：Hann峰值位置（N=64）
from aurora.audio import hann
w64 = hann(64)
peak_idx = int(np.argmax(w64))
assert peak_idx in [31, 32], f"N=64 Hann 峰值应在 31 或 32，得到 {peak_idx}"
print(f"Q3 ✅  Hann(N=64) 峰值在 idx={peak_idx}（N//2-1={64//2-1} 或 N//2={64//2}）")

# 问4：频谱泄漏 — 矩形旁瓣 vs Hann（数值验证）
from aurora.audio import sine
N = 256
sig = sine(5.5, duration=N/1000, sample_rate=1000)  # 非整数频率
rect_spec = np.abs(np.fft.rfft(sig * np.ones(N)))
hann_spec = np.abs(np.fft.rfft(sig * w64[:N] if len(w64)>=N else sig * hann(N)))
# 旁瓣：主瓣外最大值
main_bin = int(np.argmax(rect_spec))
sidelobe_rect = np.max(np.delete(rect_spec, range(max(0,main_bin-2), min(len(rect_spec),main_bin+3))))
sidelobe_hann = np.max(np.delete(hann_spec, range(max(0,main_bin-2), min(len(hann_spec),main_bin+3))))
print(f"Q4 ✅  矩形旁瓣峰={sidelobe_rect:.3f}，Hann旁瓣峰={sidelobe_hann:.3f}")
print(f"      Hann 旁瓣比矩形小 {20*np.log10(sidelobe_rect/sidelobe_hann):.1f} dB")

# 问5：describe_window 返回值验证
try:
    from aurora.audio import blackman
    bw = blackman(64)
    result = describe_window("Blackman", bw)
    if isinstance(result, dict):
        assert result.get("first") is not None
        print(f"Q5 ✅  describe_window 返回 {list(result.keys())}（Blackman 旁瓣更强，主瓣更宽）")
    else:
        print(f"Q5 ✅  describe_window 返回了结果（Blackman 旁瓣抑制最强，代价是主瓣最宽）")
except NotImplementedError:
    print(f"Q5 ✅  Blackman 旁瓣-58dB >> Hann -31dB，代价：主瓣宽度增加约 50%（分辨率下降）")
print("\n🎉 窗函数白板挑战通过！加窗=信号×权重，减少截断边缘的频谱污染。")

In [ ]:
# ✏️ 本课自评
l36_review = {
    "windowing_motivation":      None,  # 理解为什么截断信号需要加窗（减少旁瓣）？True/False
    "hann_formula":              None,  # 记住 Hann: 0.5·(1-cos(2πk/N))？True/False
    "describe_window_impl":      None,  # describe_window 实现并通过断言？True/False
    "tradeoff_sidelobe_mainlobe": None, # 理解旁瓣↓ vs 主瓣宽度↑ 的权衡？True/False
    "whiteboard_passed":         None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l36_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l36_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L36 全部通关！进入 L37：DFT 暴力实现')

---

→ **下一课**　[L37 · DFT 暴力实现](L37_dft.ipynb)

> 下节课将学习 **DFT 暴力实现**：X[k]=Σ x[n]e^{-2πikn/N}，O(N²) 双循环 + numpy 对齐验证。